# 教師なし学習

## クラスタリング
- 多数の参加者に，自分にとって「かわいいもの」と，それらに 17個の形容語がどのくらい当てはまるかを -2～2 の5段階で評価してもらったデータを元に，「かわいいもの」を2つのクラスタに分けてみる (2012年に調査したデータを一部抜粋・修正)
- クラスタリングの方法は，ユークリッド距離を使った k-means 法を用いる．

### データの読み込み

In [1]:
import pandas as pd

# kawaii.csv を読み込んで表示
kawaii = pd.read_csv("kawaii.csv", )
display(kawaii)

,評定者,対象,小さい,綺麗,癒し,柔らかい,ふわふわ,あたたかい,お洒落,美しい,優しい,女の子らしい,素敵,丸い,きらきら,無邪気,和み,幼い,華やか
0,yO+7FddMs,妹,2,-2,2,2,2,2,-1,-1,1,1,2,2,2,2,2,2,-2
1,fEkFadipa,動物,-1,2,2,0,0,2,2,2,2,-1,2,0,0,2,2,0,0
2,Zxjd7f3yE,ねこ,0,0,2,2,2,2,0,0,0,0,0,0,0,0,2,0,0
3,AQyJT8wuY,女の子,1,2,2,2,2,2,2,2,2,2,-1,1,2,2,2,0,2
4,E2PLla6IW,ハムスター,2,-1,2,2,2,2,-1,-1,1,-1,2,2,2,2,2,2,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,82g9anDgk,服・小物,-1,2,0,-2,-2,-2,2,1,-2,1,2,-2,2,-2,-1,-2,2
252,82g9anDgk,芸能人,-2,2,-1,-2,-2,-2,2,2,0,2,2,-2,2,0,-2,-1,2
253,bAAu5TQlW,飼い犬,-2,1,2,2,-1,2,-2,1,1,-2,1,-1,1,2,2,-1,-1
254,bAAu5TQlW,赤ちゃん,2,-2,2,2,0,2,-2,-2,0,1,-2,2,2,2,2,2,-2


### クラスタリングの実行
- 形容語のデータを元に，2つのクラスタに分ける

In [ ]:
# KMeans の memory leak の回避
import os
os.environ["OMP_NUM_THREADS"] = "1"

# 形容語の評定値のデータを kawaii_feat に代入
kawaii_feat = kawaii.drop(["評定者", "対象"], axis=1)

# kawaii_feat を用いてクラスタリングを実行(クラスタ数2)
from sklearn.cluster import KMeans
kawaii_cluster = KMeans(2, random_state=0).fit(kawaii_feat)

print("完了")

### 各クラスタの重心の表示
- 各クラスタに属する対象の，形容語ごとの評定値の平均 (= 重心) を求める  
  → 各クラスタの特徴を表す (-2: 当てはまらない，0: どちらともいえない, 2: 当てはまる)

In [ ]:
# 各クラスタの重心 (評定値の平均) の表示
centroid = kawaii_cluster.cluster_centers_
df_centroid = pd.DataFrame(centroid.transpose(), index=kawaii_feat.columns)
display(df_centroid)

### データに列を追加する
- `cluster`: 各対象が属するクラスタ
- `dist0`: この対象 と クラスタ0の重心 とのユークリッド距離
- `dist1`: この対象 と クラスタ1の重心 とのユークリッド距離

In [ ]:
# 元のデータ(kawaii)に cluster, dist0, dist1 の列を追加
from math import sqrt
kawaii["cluster"] = kawaii_cluster.labels_
kawaii["dist0"] = ((kawaii_feat - centroid[0])**2).apply(sum, axis = 1).map(sqrt)
kawaii["dist1"] = ((kawaii_feat - centroid[1])**2).apply(sum, axis = 1).map(sqrt)

# 結果の表を表示
display(kawaii)

### 各クラスタに属する対象を表示
- 各クラスタごとに，重心に近い方から30個を表示

In [ ]:
# 各クラスターの対象の，重心に近い方から30個を表示
print("クラスター0: ", kawaii[kawaii["cluster"]==0].sort_values("dist0").head(30)["対象"].values)
print("クラスター1: ", kawaii[kawaii["cluster"]==1].sort_values("dist1").head(30)["対象"].values)